# RAG Ground-Truth Builder

This notebook builds an evaluation set (ground-truth Q&A pairs) for the
YouTube-lecture RAG bot.

Pipeline:
1. Read the list of lectures from `./data/english_lectures.csv`.
2. Fetch each video's transcript (same logic as `app/watcher/local_watcher.py`).
3. Ask a strong LLM (Amazon Bedrock, AWS-Free-Tier-eligible) to generate 10
   question/answer pairs per video, following the ground-truth prompt spec.
4. Save everything to `./data/questions.json` in the reviewer-ready schema.

> ⚠️ Generated ground truth is **not** verified. The manual review step at
> the end of this notebook is not optional — see the reminder cell.

In [1]:
!pip install -q --upgrade youtube-transcript-api langchain-aws boto3 pandas

ERROR: Operation cancelled by user
^C


In [2]:
import os
import re
import json
import time
from dataclasses import dataclass
from urllib.parse import urlparse, parse_qs

import pandas as pd
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound


## 1. Transcript fetching

Mirrors `app/watcher/local_watcher.py::fetch_transcript_locally` so this
notebook has zero dependency on the AWS-side bot module. It auto-detects a
manually-created transcript first, falling back to auto-generated.

In [3]:
@dataclass
class Segment:
    text: str
    start: float
    duration: float


def segments_from_fetched(fetched) -> list[Segment]:
    """Converts the youtube-transcript-api FetchedTranscript into our Segment list."""
    return [Segment(text=s.text, start=s.start, duration=s.duration) for s in fetched]


def fetch_transcript_locally(video_id: str) -> tuple[str, list[Segment], str, str, str]:
    """Auto-detects and fetches the best available transcript for a video —
    same logic as bot.py's get_transcript(), duplicated here deliberately so
    this script has zero dependency on the AWS-side bot module.

    Lists every transcript available for the video and prefers a manually
    created one (i.e. provided by the uploader, not auto-generated) in
    whichever language exists; falls back to the first auto-generated one
    otherwise. No language needs to be requested or configured.

    Returns (full_text, segments, kind, lang_code, status).
    """
    api = YouTubeTranscriptApi()

    transcript_list = list(api.list(video_id))
    if not transcript_list:
        raise NoTranscriptFound(video_id, [], {})

    manual = [t for t in transcript_list if not t.is_generated]
    chosen = manual[0] if manual else transcript_list[0]

    fetched = chosen.fetch()
    segments = segments_from_fetched(fetched)
    full_text = " ".join(s.text for s in segments)

    kind = "manual" if not chosen.is_generated else "auto-generated"
    status = (
        f"\u2705 Transcript loaded ({len(segments)} segments) \u2014 "
        f"language: {chosen.language} [{chosen.language_code}], {kind}"
    )
    return full_text, segments, kind, chosen.language_code, status


def extract_video_id(url: str) -> str | None:
    """Pulls a YouTube video id out of watch/short/embed/youtu.be URLs.
    Returns None for non-YouTube sources (e.g. oyc.yale.edu) — those have no
    transcript API and are skipped upstream with a warning."""
    parsed = urlparse(url)
    host = parsed.netloc.lower()

    if "youtu.be" in host:
        return parsed.path.lstrip("/").split("/")[0] or None

    if "youtube.com" in host:
        if parsed.path == "/watch":
            qs = parse_qs(parsed.query)
            return qs.get("v", [None])[0]
        m = re.match(r"^/(embed|shorts|live)/([^/?]+)", parsed.path)
        if m:
            return m.group(2)

    return None  # not a YouTube URL

## 2. Load the lecture list

Expected CSV format (`./data/english_lectures.csv`):

```csv
link,title,duration
https://oyc.yale.edu/psychology/psyc-110/lecture-1,Introduction to Psychology \u2014 Yale,55 min
https://www.youtube.com/watch?v=kBdfcR-8hEY,Justice \u2014 The Moral Side of Murder \u2014 Harvard,55 min
```

In [8]:
CSV_PATH = "./data/english_lectures.csv"

df = pd.read_csv(CSV_PATH)
df

,link,title,duration
0,https://www.youtube.com/watch?v=rUQHPm7xfpY,"A 300,000-Year History of Human Evolution - Ro...",~1 hour
1,https://youtu.be/dvvOi_nUCRM,Black Holes and Bangs - Chris Lintott,~1 hour
2,https://www.youtube.com/watch?v=Qi4KBoirchI,Dyslexia and Language - Disorder or Difference...,~1 hour
3,https://www.youtube.com/watch?v=UrRgkK_sEUs,From Thin Air: Minimalist and process music fr...,~1 hour
4,https://www.youtube.com/watch?v=fFCDvsq6N5g,Does having a big brain make you smarter? - Al...,~55 mins
5,https://www.youtube.com/watch?v=IjfzEbHmWZ8,"Democracy: Ancient Models, Modern Challenges -...",~58 mins
6,https://www.youtube.com/watch?v=15156589,Evolution Tomorrow and Beyond - Robin May,~45 mins
7,https://www.youtube.com/watch?v=19372217,Should We Manipulate People’s Emotions? - Robi...,46 mins
8,https://www.youtube.com/watch?v=19837488,"The Sky at Night, Tonight - Chris Lintott",~1 hour
9,https://www.youtube.com/watch?v=AvVZevycQZI,"Logarithms: Mobile Phones, Modelling & Statist...",~52 mins


In [9]:
def parse_minutes(duration_str: str) -> int | None:
    m = re.search(r"(\d+)", str(duration_str))
    return int(m.group(1)) if m else None


lectures = []

for _, row in df.iterrows():
    url = row["link"]
    title = row["title"]
    duration = row.get("duration", "")

    video_id = extract_video_id(url)
    if video_id is None:
        print(f"\u26a0\ufe0f  Skipping non-YouTube source (no transcript API available): {url}")
        continue

    try:
        full_text, segments, kind, lang_code, status = fetch_transcript_locally(video_id)
        print(f"{title}: {status}")
        lectures.append({
            "video_id": video_id,
            "url": url,
            "title": title,
            "minutes": parse_minutes(duration),
            "language": lang_code,
            "transcript_kind": kind,
            "transcript": full_text,
            "segments": segments,
        })
    except NoTranscriptFound:
        print(f"\u274c No transcript available for {title} ({url})")
    except Exception as e:
        print(f"\u274c Failed to fetch transcript for {title} ({url}): {e}")

print(f"\nLoaded transcripts for {len(lectures)}/{len(df)} lectures.")

A 300,000-Year History of Human Evolution - Robin May: ✅ Transcript loaded (1745 segments) — language: English (auto-generated) [en], auto-generated
Black Holes and Bangs - Chris Lintott: ✅ Transcript loaded (480 segments) — language: English (auto-generated) [en], auto-generated
Dyslexia and Language - Disorder or Difference? - Maggie Snowling CBE: ✅ Transcript loaded (1440 segments) — language: English (auto-generated) [en], auto-generated
From Thin Air: Minimalist and process music from Africa to Arvo Pärt - Milton Mermikides: ✅ Transcript loaded (969 segments) — language: English (auto-generated) [en], auto-generated
Does having a big brain make you smarter? - Alain Goriely: ✅ Transcript loaded (181 segments) — language: English (auto-generated) [en], auto-generated
Democracy: Ancient Models, Modern Challenges - Melissa Lane: ✅ Transcript loaded (1336 segments) — language: English (auto-generated) [en], auto-generated
❌ Failed to fetch transcript for Evolution Tomorrow and Beyond -

## 3. Configure the LLM \u2014 Amazon Bedrock (AWS Free Tier)

We use **Amazon Bedrock** so no local model/GPU is needed. Among the models
covered by the **AWS Free Tier** for Bedrock (a limited pool of free tokens
for new accounts, valid for the first months), the **Amazon Nova** family is
the strongest option \u2014 and `amazon.nova-pro-v1:0` is the strongest model
in that family, with a long context window that comfortably fits a full
~55-minute lecture transcript in one call. `amazon.nova-lite-v1:0` (used as
the default in the original bot config) is kept below as a cheaper fallback
if you hit the free-tier token cap.

> Requires AWS credentials to already be configured in the environment
> (env vars, shared credentials file, or an instance/role profile) with
> Bedrock model access enabled for the chosen model in `AWS_REGION`.

In [10]:
AWS_REGION        = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_LLM_MODEL = os.getenv("BEDROCK_LLM_MODEL", "amazon.nova-pro-v1:0")   # strong, free-tier-eligible
BEDROCK_LLM_FALLBACK_MODEL = os.getenv("BEDROCK_LLM_FALLBACK_MODEL", "amazon.nova-lite-v1:0")


def build_llm(model_id: str = BEDROCK_LLM_MODEL):
    """Returns a Bedrock chat model via langchain-aws."""
    from langchain_aws import ChatBedrockConverse
    print(f"\U0001f7e0 LLM: Bedrock / {model_id}")
    return ChatBedrockConverse(
        model=model_id,
        region_name=AWS_REGION,
        temperature=0.1,
        max_tokens=4096,   # generous: 10 Q&A pairs with evidence quotes
    )


llm = build_llm()

/home/roman/Desktop/youtube_rag_bot/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🟠 LLM: Bedrock / amazon.nova-lite-v1:0


## 4. Ground-truth generation prompt

The prompt spec (paste transcript, get back a JSON array of 10 pairs:
3 factual, 2 summary, 2 multi-hop, 2 follow-up, 1 off-topic).

In [11]:
PROMPT_TEMPLATE = """You are helping build an evaluation set for a retrieval-augmented question
answering system over YouTube transcripts.

Below is the full transcript of one video, video id "{video_id}".

Produce exactly 10 question\u2013answer pairs as a JSON array. Use these types and
counts:

- 3 x `factual` \u2014 answerable from a single short passage; prefer questions whose
  answer is a specific number, name, date or term.
- 2 x `summary` \u2014 require condensing a substantial part of the video.
- 2 x `multi_hop` \u2014 require combining two passages that are at least ten minutes
  apart in the video. State both timecodes/fragments in `evidence`.
- 2 x `follow_up` \u2014 short, elliptical continuations ("say more about that",
  "why?") that only make sense after the preceding question. The first must
  follow the last `multi_hop` question, the second must follow the first
  `follow_up`.
- 1 x `off_topic` \u2014 plausible-sounding but not discussed in the video at all.

For every pair return an object shaped like:

{{
  "id": "{video_id}-q<N>",
  "type": "<one of the types above>",
  "follow_up_to": "<id of the preceding question, or null>",
  "question": "<the question, in the language of the video>",
  "ground_truth": "<a complete answer, 1-3 sentences, using only the transcript>",
  "evidence": "<the transcript fragment(s) and timecode(s) that support it>"
}}

Rules:
- Never use knowledge from outside the transcript.
- Do not paraphrase the transcript so heavily that the wording of the question
  gives away which chunk contains the answer \u2014 that biases lexical retrieval.
- For the `off_topic` item, `ground_truth` must state that the video does not
  cover the subject.
- Return ONLY the JSON array, nothing else \u2014 no markdown fences, no preamble.

TRANSCRIPT:
{transcript}
"""

In [12]:
def _strip_code_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```(json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def generate_ground_truth(video_id: str, transcript: str, llm, retries: int = 2) -> list[dict]:
    """Calls the LLM with the ground-truth prompt and parses the JSON array it returns."""
    prompt = PROMPT_TEMPLATE.format(video_id=video_id, transcript=transcript)

    last_err = None
    for attempt in range(1, retries + 2):
        response = llm.invoke(prompt)
        raw = response.content if isinstance(response.content, str) else str(response.content)
        cleaned = _strip_code_fences(raw)
        try:
            parsed = json.loads(cleaned)
            if not isinstance(parsed, list):
                raise ValueError("Model did not return a JSON array")
            return parsed
        except (json.JSONDecodeError, ValueError) as e:
            last_err = e
            print(f"  \u26a0\ufe0f  Attempt {attempt} failed to parse JSON for {video_id}: {e}")
            time.sleep(1)

    raise RuntimeError(f"Could not get valid JSON for {video_id} after {retries + 1} attempts") from last_err

## 5. Run generation for each lecture and assemble the output

In [13]:
videos_out = []

for lec in lectures:
    print(f"Generating ground truth for: {lec['title']} ({lec['video_id']})")
    try:
        questions = generate_ground_truth(lec["video_id"], lec["transcript"], llm)
    except RuntimeError as e:
        print(f"  \u274c {e} \u2014 skipping this video.")
        continue

    videos_out.append({
        "video_id": lec["video_id"],
        "url": lec["url"],
        "title": lec["title"],
        "language": lec["language"],
        "minutes": lec["minutes"],
        "questions": questions,
    })

print(f"\nGenerated ground truth for {len(videos_out)}/{len(lectures)} lectures.")

Generating ground truth for: A 300,000-Year History of Human Evolution - Robin May (rUQHPm7xfpY)
Generating ground truth for: Black Holes and Bangs - Chris Lintott (dvvOi_nUCRM)
Generating ground truth for: Dyslexia and Language - Disorder or Difference? - Maggie Snowling CBE (Qi4KBoirchI)
Generating ground truth for: From Thin Air: Minimalist and process music from Africa to Arvo Pärt - Milton Mermikides (UrRgkK_sEUs)
Generating ground truth for: Does having a big brain make you smarter? - Alain Goriely (fFCDvsq6N5g)
Generating ground truth for: Democracy: Ancient Models, Modern Challenges - Melissa Lane (IjfzEbHmWZ8)
Generating ground truth for: Logarithms: Mobile Phones, Modelling & Statistics? - Oliver Johnson (AvVZevycQZI)

Generated ground truth for 7/7 lectures.


In [14]:
OUTPUT_PATH = "./data/questions.json"

output = {
    "_comment": (
        f"{len(videos_out)} video(s) x 10 questions. Question types: factual (3), "
        "summary (2), multi_hop (2), follow_up (2), off_topic (1). A follow_up "
        "question MUST set follow_up_to to the id of the question it follows, and "
        "must come directly after it \u2014 that is what exercises the router. "
        "GENERATED, NOT YET MANUALLY VERIFIED \u2014 see the review checklist below."
    ),
    "videos": videos_out,
}

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved {sum(len(v['questions']) for v in videos_out)} questions across {len(videos_out)} videos to {OUTPUT_PATH}")

Saved 70 questions across 7 videos to ./data/questions.json


## 6. Manual review \u2014 required before using this as ground truth

Generated pairs are a draft, not ground truth. For every pair in
`./data/questions.json`, check:

1. **Is the ground truth actually in the transcript?** Open the
   timecode/fragment in `evidence` and confirm.
2. **Is `factual` really single-hop, and `multi_hop` really two-hop** (i.e.
   from passages \u2265 10 minutes apart)?
3. **Does each `follow_up` genuinely make no sense standalone?** If it does,
   the router test it's meant to exercise is worthless.
4. **Is the wording natural** \u2014 the way a Telegram user would actually type
   it, not the way a dataset builder would?

Only after this pass should `questions.json` be treated as a trustworthy
evaluation set.

In [ ]:
print(
    "Reminder: the questions in ./data/questions.json are LLM-generated drafts.\n"
    "Open each 'evidence' field against the source transcript and fix any pair\n"
    "that fails the 4 checks above before using this as an evaluation set."
)